In [ ]:
import json
import random
from pathlib import Path
from collections import Counter, defaultdict

# ----------------------------
# Paths
# ----------------------------

PROJECT_ROOT = Path("/Users/tildeidunsloth/Desktop/Thesis")
DATA_DIR = PROJECT_ROOT / "data/cleaned"


sci_fi_data_path = DATA_DIR / "sci_fi_stories_cleaned.jsonl"
romance_data_path = DATA_DIR / "romance_stories_cleaned.jsonl"
literary_fiction_data_path = DATA_DIR / "lit_fiction_stories_cleaned.jsonl"

FILES = [
    sci_fi_data_path,
    romance_data_path,
    literary_fiction_data_path
]

# ----------------------------
# Configuration
# ----------------------------

N_TOTAL = 100  # total number of stories to evaluate

VALID_LABELS = {
    "1": "sci-fi",
    "2": "romance",
    "3": "literary fiction"
}

# ----------------------------
# Load JSONL
# ----------------------------

def load_jsonl(filepath):
    stories = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            stories.append(json.loads(line))
    return stories


all_stories = []

for file in FILES:
    data = load_jsonl(file)
    all_stories.extend(data)

print(f"Loaded {len(all_stories)} total stories.")


# ----------------------------
# Balanced sampling
# ----------------------------

by_genre = defaultdict(list)
for item in all_stories:
    by_genre[item["genre"]].append(item)

n_per_genre = N_TOTAL // 3

sampled = []
for genre, stories in by_genre.items():
    sampled.extend(random.sample(stories, min(n_per_genre, len(stories))))

# Fill remainder if needed
while len(sampled) < N_TOTAL:
    sampled.append(random.choice(all_stories))

random.shuffle(sampled)


# ----------------------------
# Evaluation loop
# ----------------------------

results = []

for i, item in enumerate(sampled, 1):
    print("\n" + "="*80)
    print(f"Story {i}/{N_TOTAL}")
    print("="*80)
    print(item["story"])
    print("\nGuess the genre:")
    print("1 = Sci-fi")
    print("2 = Romance")
    print("3 = Literary Fiction")

    guess = None
    while guess not in VALID_LABELS:
        guess = input("Your guess (1/2/3): ").strip()

    predicted = VALID_LABELS[guess]
    true_genre = item["genre"]

    results.append({
        "predicted": predicted,
        "true": true_genre
    })


# ----------------------------
# Evaluation metrics
# ----------------------------

correct = sum(1 for r in results if r["predicted"] == r["true"])
accuracy = correct / len(results)

print("\n" + "="*80)
print("RESULTS")
print("="*80)
print(f"Accuracy: {accuracy:.2%} ({correct}/{len(results)})")

# Confusion matrix
confusion = defaultdict(Counter)
for r in results:
    confusion[r["true"]][r["predicted"]] += 1

genres = ["fantasy", "romance", "literary fiction"]

print("\nConfusion Matrix (rows = true, columns = predicted):\n")

print(f"{'':20}", end="")
for g in genres:
    print(f"{g:20}", end="")
print()

for true in genres:
    print(f"{true:20}", end="")
    for pred in genres:
        print(f"{confusion[true][pred]:20}", end="")
    print()

Loaded 300 total stories.

Story 1/100
When Mara first took the job at Stonebridge Correctional Facility, she told herself it would be temporary a steady paycheck while she finished school loans, a place to keep a steady watch over a life that had lately felt unmoored. She kept her hair tightly braided, her scrubs ironed, and a small brass locket at her throat that had belonged to her grandmother. Inside was a sepia photograph of two young women with braids and quiet smiles. The locket was thin, warm against her sternum, and every time she felt its edge she felt steadier.

On a rain-slimmed Tuesday when a thermodynamic furnace hummed under the steel and magnesium of the prison, Mara unlatched a cell door to attend to a man suffering palpitations. The lights in Block C flickered. The man gripped the rails; he smelled like machine oil and stale bread. She stilled his hand, clicked the monitor into place, checked his pulse. The room moved with the rhythm of alarms and the clack of keys.



KeyboardInterrupt: Interrupted by user

In [15]:
import json
import random
from pathlib import Path
from collections import Counter, defaultdict

# ----------------------------
# Paths
# ----------------------------

PROJECT_ROOT = Path("/Users/tildeidunsloth/Desktop/Thesis")
DATA_DIR = PROJECT_ROOT / "data/cleaned"
EVAL_DIR = PROJECT_ROOT / "notebooks/Story_Quality_Checks"

sci_fi_data_path = DATA_DIR / "sci_fi_stories_cleaned.jsonl"
romance_data_path = DATA_DIR / "romance_stories_cleaned.jsonl"
literary_fiction_data_path = DATA_DIR / "lit_fiction_stories_cleaned.jsonl"

FILES = [
    sci_fi_data_path,
    romance_data_path,
    literary_fiction_data_path
]

SAMPLED_FILE = EVAL_DIR / "sampled_stories.json"
RESULTS_FILE = EVAL_DIR / "evaluation_results.json"

N_TOTAL = 100
BATCH_SIZE = 10

VALID_LABELS = {
    "1": "science fiction",
    "2": "romance",
    "3": "literary fiction"
}

# ----------------------------
# Utilities
# ----------------------------

def load_jsonl(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def save_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

def load_json(path):
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return None


# ----------------------------
# Step 1: Create fixed sample (only once)
# ----------------------------

if not SAMPLED_FILE.exists():
    print("Creating new 100-story sample...")

    all_stories = (
        load_jsonl(sci_fi_data_path) +
        load_jsonl(romance_data_path) +
        load_jsonl(literary_fiction_data_path)
    )

    by_genre = defaultdict(list)
    for item in all_stories:
        by_genre[item["genre"]].append(item)

    n_per_genre = N_TOTAL // 3
    sampled = []

    for genre, stories in by_genre.items():
        sampled.extend(random.sample(stories, min(n_per_genre, len(stories))))

    while len(sampled) < N_TOTAL:
        sampled.append(random.choice(all_stories))

    random.shuffle(sampled)

    save_json(sampled, SAMPLED_FILE)
else:
    print("Using existing sampled stories.")

sampled = load_json(SAMPLED_FILE)


# ----------------------------
# Step 2: Load previous results
# ----------------------------

results = load_json(RESULTS_FILE) or []

already_done = len(results)
remaining = N_TOTAL - already_done

print(f"\nAlready completed: {already_done}")
print(f"Remaining: {remaining}")

if remaining == 0:
    print("All 100 stories already evaluated.")
else:
    to_do_now = min(BATCH_SIZE, remaining)

    print(f"\nStarting batch of {to_do_now} stories...")

    for i in range(already_done, already_done + to_do_now):
        item = sampled[i]

        print("\n" + "="*80)
        print(f"Story {i+1}/{N_TOTAL}")
        print("="*80)
        print(item["story"])
        print("\nGuess the genre:")
        print("1 = Science Fiction")
        print("2 = Romance")
        print("3 = Literary Fiction")

        guess = None
        while guess not in VALID_LABELS:
            guess = input("Your guess (1/2/3): ").strip()

        predicted = VALID_LABELS[guess]
        true_genre = item["genre"]

        results.append({
            "predicted": predicted,
            "true": true_genre
        })

        save_json(results, RESULTS_FILE)

    print("\nBatch complete. You can safely stop now and resume later.")


# ----------------------------
# Step 3: If finished, compute accuracy
# ----------------------------

if len(results) == N_TOTAL:

    correct = sum(1 for r in results if r["predicted"] == r["true"])
    accuracy = correct / len(results)

    print("\n" + "="*80)
    print("FINAL RESULTS")
    print("="*80)
    print(f"Accuracy: {accuracy:.2%} ({correct}/{len(results)})")

    confusion = defaultdict(Counter)
    for r in results:
        confusion[r["true"]][r["predicted"]] += 1

    genres = ["science fiction", "romance", "literary fiction"]

    print("\nConfusion Matrix (rows = true, columns = predicted):\n")

    print(f"{'':20}", end="")
    for g in genres:
        print(f"{g:20}", end="")
    print()

    for true in genres:
        print(f"{true:20}", end="")
        for pred in genres:
            print(f"{confusion[true][pred]:20}", end="")
        print()

Using existing sampled stories.

Already completed: 100
Remaining: 0
All 100 stories already evaluated.

FINAL RESULTS
Accuracy: 89.00% (89/100)

Confusion Matrix (rows = true, columns = predicted):

                    science fiction     romance             literary fiction    
science fiction                       30                   0                   3
romance                                1                  31                   2
literary fiction                       1                   4                  28
